## Convert a .grd to GeoTIFF
- `.grd` files produced by gmt sometimes have weirdness in the longitude
- The most basic is converting, say -90 to 270 but some other strange stuff happens at times
- Sometimes, here it will say that the longitude is, say 30 deg, and when you load into GIS it's still 30 deg, but off by 360 deg. In this case, we just have to add (or maybe subtract in some cases?) 360 deg until it magically work. There has to be a better way but I haven't found one yet. 

In [2]:
# ! cd /Volumes/T9/2025-01-07_tibet/S1/F1/intf/2025004_2025016/
# ! ls
! proj_ra2ll.csh /Volumes/T9/2025-01-07_tibet/S1/F1/intf/2025004_2025016/trans.dat /Volumes/T9/2025-01-07_tibet/S1/F1/intf/2025004_2025016/yphase.grd /Volumes/T9/2025-01-07_tibet/S1/F1/intf/2025004_2025016/yphase_ll.grd

proj_ra2ll.csh
dyld[97110]: Library not loaded: @rpath/libgfortran.5.dylib
  Referenced from: <C435405B-BFF1-399F-A1EC-BD43418F546A> /Users/hyin/miniforge3/envs/pygmt/lib/libopenblas.0.dylib
  Reason: tried: '/Users/hyin/miniforge3/envs/pygmt/lib/libgfortran.5.dylib' (duplicate LC_RPATH '@loader_path'), '/Users/hyin/miniforge3/envs/pygmt/lib/libgfortran.5.dylib' (duplicate LC_RPATH '@loader_path'), '/Users/hyin/miniforge3/envs/pygmt/lib/libgfortran.5.dylib' (duplicate LC_RPATH '@loader_path'), '/Users/hyin/miniforge3/envs/pygmt/bin/../lib/libgfortran.5.dylib' (duplicate LC_RPATH '@loader_path'), '/Users/hyin/miniforge3/envs/pygmt/bin/../lib/libgfortran.5.dylib' (duplicate LC_RPATH '@loader_path'), '/usr/local/lib/libgfortran.5.dylib' (no such file), '/usr/lib/libgfortran.5.dylib' (no such file, not in dyld cache)
Abort
dyld[97113]: Library not loaded: @rpath/libgfortran.5.dylib
  Referenced from: <C435405B-BFF1-399F-A1EC-BD43418F546A> /Users/hyin/miniforge3/envs/pygmt/lib/libopenblas.0

In [ ]:

import xarray as xr
import rioxarray
import os

merge_dir = '/Volumes/T9/2025-mendocino-mystery/alos2/A069_800/intf/2022317_2022359/'
filetypes = [ "corr_ll", "los_ll", "los_ll_dtr", "xphase_mask_ll", "yphase_mask_ll", "phasefilt_ll","phasefilt_mask_ll"]    # "los_ll", "xphase_mask_ll", "yphase_mask_ll", "phasefilt_ll"

# for file in filetypes:
#     file_path = merge_dir + file + '.grd'
#     if not os.path.exists(file_path):
#         print(f"File not found: {file_path}")
#         print(f"File not found: {file_path}")
#         continue  # Skip to the next file if it doesn't exist
#     ds = xr.open_dataset(merge_dir + file +'.grd')
#         # Check if any longitude values are negative
#     if (ds.lon < 0).any():
#         print("Negative longitude values found. Converting to positive.")
#     # ds = ds.assign_coords(lon=ds.lon - 360)
#     ds = ds.assign_coords(lon=ds.lon + 360)
#     ds = ds.rename({"lon": "x", "lat":"y"})
#     # da = ds.to_array()
#     # da = da.drop_vars(["spatial_ref"], errors="ignore")
#     # da = da.drop_vars(["band"], errors="ignore")

#     ds.rio.write_crs("EPSG:4326", inplace=True)  # WGS 84
#     # da = da.squeeze("band")
#     # print(da.shape)
#     ds.rio.to_raster(merge_dir + file + '.tiff')
#     # print(ds)

for file in filetypes:
    file_path = merge_dir + file + '.grd'
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue  

    ds = xr.open_dataset(file_path)

    # --- Normalize longitude to -180..180 ---
    if "lon" in ds.coords:
        lon = ds.lon.values
        # Wrap into [-180, 180]
        lon_wrapped = ((lon + 180) % 360) - 180
        ds = ds.assign_coords(lon=lon_wrapped)

        # Ensure sorted lon (important for raster writing!)
        ds = ds.sortby("lon")

    ds = ds.rename({"lon": "x", "lat": "y"})

    ds.rio.write_crs("EPSG:4326", inplace=True)  # WGS 84
    out_file = os.path.join(merge_dir, file + '.tiff')
    ds.rio.to_raster(out_file)

    print(f"Wrote {out_file}")


Negative longitude values found. Converting to positive.
File not found: /Volumes/T9/2025-mendocino-mystery/alos2/A069_800/intf/2022317_2022359/los_ll.grd
File not found: /Volumes/T9/2025-mendocino-mystery/alos2/A069_800/intf/2022317_2022359/los_ll.grd
File not found: /Volumes/T9/2025-mendocino-mystery/alos2/A069_800/intf/2022317_2022359/los_ll_dtr.grd
File not found: /Volumes/T9/2025-mendocino-mystery/alos2/A069_800/intf/2022317_2022359/los_ll_dtr.grd
Negative longitude values found. Converting to positive.
Negative longitude values found. Converting to positive.
Negative longitude values found. Converting to positive.
Negative longitude values found. Converting to positive.


## Phasefilt grd

<xarray.DataArray 'lon' (lon: 3790)> Size: 30kB
array([285.076736, 285.077431, 285.078125, ..., 287.706597, 287.707292,
       287.707986], shape=(3790,))
Coordinates:
  * lon      (lon) float64 30kB 285.1 285.1 285.1 285.1 ... 287.7 287.7 287.7
Attributes:
    long_name:      longitude
    units:          degrees_east
    standard_name:  longitude
    axis:           X
    actual_range:   [285.07638889 287.70833333]